In [3]:
import pandas as pd
import math
import re

# Zero-Shot Topic Generation for BERTopic Model

**Methodology Note:** The following zero-shot topics were generated to guide the BERTopic model. To ensure the topics were highly relevant to this specific dataset, a random sample of 1,000 guest reviews was extracted and processed using a Large Language Model.

**Model Used:** ChatGPT Plus (GPT-5.4 Thinking Mode - Standard Effort)

### The Prompt
> Act as an expert hospitality data analyst. I have uploaded a CSV containing a random sample of 1,000 hotel guest reviews. I am building a BERTopic model and need to generate zero-shot topics.
> 
> Please read the entire document carefully. Based strictly on the themes present in the text of these reviews, generate two lists for me:
> 1. A list of 10 Major Topics (broad, high-level categories like 'Service' or 'Room Quality').
> 2. A list of 20 Minor Topics (specific, granular issues or highlights).
>
> Do not use booking metadata (like 'couple' or 'business trip') as topics. Focus exclusively on the actual sentiments and complaints the guests are writing about.

### The AI-Generated Output

#### 10 Major Topics
1. Staff Service
2. Room Comfort & Quality
3. Cleanliness
4. Location & Accessibility
5. Breakfast & Food
6. Bathroom & Shower Experience
7. Noise & Sleep Disturbance
8. Facilities & Amenities
9. Value for Money
10. Maintenance & Room Condition

#### 20 Minor Topics
1. Helpful and friendly staff
2. Reception and check-in/check-out experience
3. Comfortable beds and pillows
4. Small room size
5. Spacious room layout
6. Room cleanliness and housekeeping
7. Dirty or poorly cleaned rooms
8. Bathroom size and layout
9. Shower quality and water pressure
10. Air conditioning / room temperature control
11. Wi-Fi quality and reliability
12. Breakfast quality
13. Breakfast variety and choice
14. Restaurant / bar food and service
15. Quiet rooms and peaceful stay
16. Noise from street, hallway, or neighbouring rooms
17. Proximity to city centre / attractions
18. Access to public transport (station / metro / tube)
19. Pool / gym / spa quality
20. Old, dated, or poorly maintained furnishings and fixtures

In [4]:
# Code to randomly generate dataset of 1000 rows
# Define your file paths
input_file = "../Datasets/Hotel_Reviews.csv"
output_file = "../Datasets/Hotel_Reviews_Sample_1000.csv"

print(f"Reading full dataset from: {input_file}...")
df = pd.read_csv(input_file)

# Randomly sample 1000 rows. 
# (Setting random_state ensures you get the exact same random sample if you re-run the cell)
sample_df = df.sample(n=1000, random_state=42)

# Save the sampled data to a new CSV file
sample_df.to_csv(output_file, index=False)

print(f"Successfully saved {len(sample_df)} random reviews to: {output_file}")

Reading full dataset from: ../Datasets/Hotel_Reviews.csv...
Successfully saved 1000 random reviews to: ../Datasets/Hotel_Reviews_Sample_1000.csv


Note there's a bug/feature of the below function that's sort of the nature of natural language.
If you split by "." you might occasionally split at an important point: 
"I stocked up on supplies e.g. baked beans" would get split into "I stocked up on supplies e", "g" and "baked beans". 

Use this approach with caution and check the outputs!

In [5]:
def split_and_update_indices(test_list, test_index, split_list):
    """
    Splits elements in a list based on a list of splitters and updates the index list accordingly.
    Useful below for keeping track of who said what from topic modelling whilst splitting longer responses.

    Parameters
    test_list : list
        List of strings to split.
    test_index : list
        List of indices corresponding to the strings in test_list.
    split_list : list
        List of strings to split on.

    Returns
    new_list : list
        List of strings split on the splitters.
    new_index : list
        List of indices corresponding to the strings in new_list.
    """
    new_list = []
    new_index = []

    for element, index in zip(test_list, test_index):
        split_elements = [element]
        for splitter in split_list:
            temp_list = []
            for sub_element in split_elements:
                temp_list.extend(sub_element.split(splitter))
            split_elements = temp_list
        
        # Remove empty strings resulting from split
        split_elements = [elem for elem in split_elements if elem]

        new_list.extend(split_elements)
        new_index.extend([index] * len(split_elements))

    return new_list, new_index

# Example usage
Test_list = ["Sentence. New sentence and a bit more", "Something else", "More stuff and even more"]
Test_index = [0, 1, 2]
SplitList = [".", " and"]

New_list, New_index = split_and_update_indices(Test_list, Test_index, SplitList)

print(New_list)
print(New_index)

['Sentence', ' New sentence', ' a bit more', 'Something else', 'More stuff', ' even more']
[0, 0, 0, 1, 2, 2]


In [6]:
import pandas as pd
import math
import re

def data_loader(DataSet="Hotel", SplitterList=[". ", ", ", "-", "and", ";"], Type="Split", index="ID"):
    """
    Load in the dataset, process it into a list of data.
        
    Parameters:
    DataSet (str): The dataset to load in. Options are "Hotel".
    SplitterList (list): List of strings to split the documents by, only applies if Type is "Split".
    Type (str): The type of processing to apply to the dataset. Options are "Split" or "Combine".
    index (str): The column to use as an index for the data. Useful if you want to match documents in your topic model to the original data.
    If None, then it just uses the index from the loaded dataframe.

    If Type = "Combine" then any info from all 3 potential columns are combined into 1 entry per person.
    This appears ideal but the larger the sentence, the harder it is for BERT to deduce what sentences are similar.

    If Type = "Split" then the data is split by the SplitterList and each entry is split into multiple entries.
    A separate index is outputted which corresponds to the original data for each entry in the list 
    (this even works for the splitter option).

    Returns:
    docs (list): List of strings to use for topic modelling.
    doc_index (list): List of indices corresponding to the strings in docs.
    zero_shot_topic_list_major (list): List of major topics for zero-shot classification.
    zero_shot_topic_list_minor (list): List of minor topics for zero-shot classification.
    """
    
    if DataSet == "Hotel":
        # Hotel Data
        RawText = pd.read_csv("../Datasets/Hotel_Reviews_Sample_1000.csv")
        RawText['ID'] = RawText.index
        
        RawText['Negative_Review'] = RawText['Negative_Review'].apply(lambda x: math.nan if str(x).strip() == 'No Negative' else x)
        RawText['Positive_Review'] = RawText['Positive_Review'].apply(lambda x: math.nan if str(x).strip() == 'No Positive' else x)
        
        RawTextAnalysis = RawText[["Negative_Review", "Positive_Review"]]

        # Updated with the AI-generated topics based on the 1000 review sample
        zero_shot_topic_list_major = [
            "Staff Service", "Room Comfort & Quality", "Cleanliness", 
            "Location & Accessibility", "Breakfast & Food", "Bathroom & Shower Experience", 
            "Noise & Sleep Disturbance", "Facilities & Amenities", "Value for Money", 
            "Maintenance & Room Condition"
        ]
        zero_shot_topic_list_minor = [
            "Helpful and friendly staff", "Reception and check-in/check-out experience", 
            "Comfortable beds and pillows", "Small room size", "Spacious room layout", 
            "Room cleanliness and housekeeping", "Dirty or poorly cleaned rooms", 
            "Bathroom size and layout", "Shower quality and water pressure", 
            "Air conditioning / room temperature control", "Wi-Fi quality and reliability", 
            "Breakfast quality", "Breakfast variety and choice", "Restaurant / bar food and service", 
            "Quiet rooms and peaceful stay", "Noise from street, hallway, or neighbouring rooms", 
            "Proximity to city centre / attractions", "Access to public transport (station / metro / tube)", 
            "Pool / gym / spa quality", "Old, dated, or poorly maintained furnishings and fixtures"
        ]
    else:
        print("Invalid DataSet")
        exit()
    
    print("Data loaded: ", DataSet)
    print("Number of rows: ", RawTextAnalysis.shape[0])
    # Find how many rows have only NaNs or missing values
    missing_values_count = RawTextAnalysis.isnull().all(axis=1).sum()
    print("Number of rows with no responses: ", missing_values_count)
    print("Percentage of rows with no responses: ", missing_values_count / RawTextAnalysis.shape[0] * 100)

    #HeatTextAnalysis = HeatText[["ADVNC_TEXT", "WTHR_TEXT", "OTHR_TEXT", "FREE_TEXT"]]
    #.drop(["ID", "WTHR", "OTHR", "ALERT", "ADVNC", "HEALTH", "Health Conditions", "Symtoms1", "Symtoms2", "Symtoms3"], axis=1)
    if Type == "Split":
        data = RawTextAnalysis.values.flatten().tolist()
        dataset_size = len(data)
        # Remove NaN values from the list
        docs = [value for value in data if not (isinstance(value, float) and math.isnan(value))]
        removed_size = len(docs)

        print("List length: ", dataset_size)
        print("Removed NaN values from list: ", dataset_size - removed_size)
        print("Percentage non-responses removed: ", (dataset_size - removed_size) / dataset_size * 100)
        print("Final dataset size: ", removed_size)

        if index == False:
            doc_index = []
        else:
            doc_index = RawText[index].tolist()

        for element in docs:
            doc_index.append(data.index(element))
        
        if len(SplitterList) > 0:
            print("Length of docs before splitting: ", len(docs))
            print("Items to split by: ", SplitterList)
            docs, doc_index = split_and_update_indices(docs, doc_index, SplitterList)
            print("Length of docs after splitting: ", len(docs))

    elif Type == "Combine":
        temp_index = RawText[index]
        RawTextAnalysis = pd.merge(temp_index, RawTextAnalysis, left_index=True, right_index=True)

        dataset_size = RawTextAnalysis.shape[0]
        dataset = RawTextAnalysis.dropna(how="all", subset=RawTextAnalysis.columns[1:])

        if index == False:
            doc_index = dataset[index].tolist()
        else:
            doc_index = RawText[index]
            
        removed_size = dataset.shape[0]

        print("List length: ", dataset_size)
        print("Removed NaN values from list: ", dataset_size - removed_size)
        print("Percentage non-responses removed: ", (dataset_size - removed_size) / dataset_size * 100)
        print("Final dataset size: ", removed_size)

        docs = []

        for row in dataset.iterrows():
            values = row[1].dropna().values
            #print(values)
            value_combined = ". ".join([str(value) for value in values if value is not None])
            #print(value_combined)
            #print("")
            docs.append(value_combined)

        doclen = len(docs)    
        docs = [doc for doc in docs if re.search('[a-zA-Z]', doc)]
        if doclen != len(docs):
            print("Removed empty strings from dataset")
            print("Length of docs after processing: ", len(docs))
            
    else:
        print("Invalid Process Type")
        exit()
                     
    return docs, doc_index, zero_shot_topic_list_major, zero_shot_topic_list_minor

In [7]:
Name = "Hotel" 

docs, docs_index, zero_shot_topic_list_major, zero_shot_topic_list_minor = data_loader(Name)
print(f"Loaded {len(docs)} documents for topic modelling.")

Data loaded:  Hotel
Number of rows:  1000
Number of rows with no responses:  0
Percentage of rows with no responses:  0.0
List length:  2000
Removed NaN values from list:  314
Percentage non-responses removed:  15.7
Final dataset size:  1686
Length of docs before splitting:  1686
Items to split by:  ['. ', ', ', '-', 'and', ';']
Length of docs after splitting:  2981
Loaded 2981 documents for topic modelling.


You can remove stop works that include stuff like "see previous question" or "no answer" this only removes them from the representation of the data, not the actual data.
If you remove all the viable representation then you'll still get topic clusters but they'll just have no representation!

# Domain-Specific Stop Word Generation

**Methodology Note:** To prevent the BERTopic model from creating uninformative clusters centered around widespread, dataset-specific terms (e.g., "hotel", "room", "stay"), the standard NLTK English stop words list was used with custom domain-specific "filler" words. A random sample of 1,000 guest reviews was processed using a Large Language Model to identify the highest-frequency words that lack specific feature or sentiment value.

**Model Used:** ChatGPT Plus (GPT-5.4 Thinking Mode - Standard Effort)

### The Prompt
> Act as an expert NLP data scientist. I am building a BERTopic model to analyze hotel guest reviews and I have uploaded a sample dataset of 1,000 reviews. 
>
> I am already using the standard NLTK English stop words list, but I need to create a list of domain-specific stop words to prevent my model from creating generic, uninformative clusters.
>
> Please read through the reviews and identify the top 30 most frequent "filler" words specific to this dataset.  
>
> **Strict Constraints:**
> 1. **Include** generic nouns and verbs that appear constantly but offer no value to a specific topic (e.g., "hotel", "room", "stay", "booking", "get", "went", "told").
> 2. **DO NOT include** words that carry sentiment (e.g., "good", "bad", "dirty", "rude", "amazing").
> 3. **DO NOT include** words that represent specific features a guest might care about (e.g., "breakfast", "bed", "shower", "staff", "wifi").

### The AI-Generated Output

```python
["hotel", "room", "rooms", "stay", "stayed", "booking", "booked", "check", "night", "day", "time", "area", "place", "city", "walk", "minutes", "people", "arrival", "asked", "told", "got", "pay", "work", "located", "near", "away", "floor", "building", "way", "morning"]

In [ ]:
import nltk
from nltk.corpus import stopwords

stop_words = stopwords.words('english')

free_s_w = ["hotel", "room", "rooms", "stay", "stayed", "booking", "booked", "check", "night", "day", "time", "area", "place", "city", "walk", "minutes", "people", "arrival", "asked", "told", "got", "pay", "work", "located", "near", "away", "floor", "building", "way", "morning"]
stop_words.extend(free_s_w)

# Print stop words across several lines to make it easier to read
for i in range(0, 5):
    print(stop_words[i:i+10])

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']
['about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and']
['above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any']
['after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are']
['again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren']
numpy==1.26.4
scipy==1.17.1
scikit-learn==1.5.2
umap-learn==0.5.7
bertopic==0.17.4
sentence-transformers==5.3.0
numba==0.64.0
pynndescent==0.6.0
hdbscan==0.8.40
pandas==2.2.2
matplotlib==3.9.2
nltk==3.9.4


In [9]:
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer


# Step 1 - Extract embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

# Step 1 is by far the longest part of the process, so I recommend running this as few times as possible, especially for larger datasets

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/94 [00:00<?, ?it/s]

In [10]:
# Step 2 - Reduce dimensionality
umap_model = UMAP(n_neighbors=20, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
# High number of nearest neighbors and low minimum distance can help to preserve local structure

# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=60, min_samples=1, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# The parameter "min_cluster_size" here lets you indirectly influence the number of clusters by setting a minimum size for each cluster.
# Setting "min_samples" to a lower value can help you reduce the size of the outliers cluster.

# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words=stop_words, min_df=1, ngram_range=(1,3))

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations
representation_model = MaximalMarginalRelevance(diversity=0.7)
# 0 being not diverse, 1 being most diverse

# All steps together
BERT_model = BERTopic(
    embedding_model=embedding_model,          # Step 1 - Extract embeddings
    umap_model=umap_model,                    # Step 2 - Reduce dimensionality
    hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
    vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
    ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
    representation_model=representation_model, # Step 6 - (Optional) Fine-tune topic representations

    #zeroshot_topic_list=zero_shot_topic_list_minor, zeroshot_min_similarity=0.9, # Comment this out if you don't want zero-shot
    # 0 similarity = only use my topic lists, 1 similarity = barely pay attention to my topic lists.
)

In [11]:
topics, _ = BERT_model.fit_transform(docs)

# You can also reduce outliers after BERTopic has been fitted
new_topics = BERT_model.reduce_outliers(docs, topics)

# # Reduce outliers with pre-calculate embeddings instead
# new_topics = BERT_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings)


In [12]:
doc_inf = BERT_model.get_document_info(docs)
doc_inf["Person_id"] = docs_index
doc_inf.to_csv(f"BERTModelRawOutputs/{Name}_Document_Info.csv")
doc_inf.head()

OSError: Cannot save file into a non-existent directory: 'BERTModelRawOutputs'

In [ ]:
top_inf = BERT_model.get_topic_info()
# Calculate the number of unique person IDs per topic
unique_person_ids_per_topic = doc_inf.groupby('Topic')['Person_id'].nunique().reset_index()
unique_person_ids_per_topic.columns = ['Topic', 'Unique_Person_Count']

# Merge with the topic information dataframe
top_inf = top_inf.merge(unique_person_ids_per_topic, on='Topic', how='left')
top_inf.to_csv(f"BERTModelRawOutputs/{Name}_Topic_Info.csv")
top_inf.head()

# Topic -1 is the "outlier" topic

In [ ]:
ax = doc_inf["Name"].value_counts(ascending=True).plot(kind='barh', xlabel="Number of responses", ylabel="Topic", title="Number of responses per topic")
fig3 = ax.get_figure()
fig3.savefig(f"BERTModelRawOutputs/{Name}_Number_of_responses_per_topic.png", bbox_inches='tight')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
# Create the bar plot
ax = sns.barplot(x="Unique_Person_Count", y="Name", data=top_inf)

# Set the labels and title
ax.set(xlabel="Number of unique people", ylabel="Topic", title="Number of unique people per topic")

# Show the plot
plt.show()
fig = ax.get_figure()
fig.savefig(f"BERTModelRawOutputs/{Name}_Number_of_unique_people_per_topic.png", bbox_inches='tight')

In [ ]:
similar_topics, similarity = BERT_model.find_topics("elderly", top_n=5)
BERT_model.get_topic(similar_topics[0])

In [ ]:
BERT_model.visualize_topics()

In [ ]:
number_of_topics = len(BERT_model.get_topic_info())
fig = BERT_model.visualize_barchart(top_n_topics=number_of_topics)
fig

In [ ]:
fig2 = BERT_model.visualize_hierarchy()
fig2

In [ ]:
# topics_to_merge = [[1, 2]
#                    [3, 4]]
# BERT_model.merge_topics(docs, topics_to_merge)



#BERT_model.reduce_topics(docs, nr_topics=30)


# Topics_to_merge = []
# merge_list = []
# while True:
#     print("Current merge list: ", merge_list)
#     print("Current Topics_to_merge: ", Topics_to_merge)
#     topic = input("Enter topic numbers to merge, 'm' to conclude that merge list and 'q' to quit: ")
#     if topic == "q":
#         break
#     elif topic == "m":
#         if len(merge_list) > 1:
#             Topics_to_merge.append(merge_list)
#         merge_list = []
#     else:
#         merge_list.append(int(topic))
# if len(Topics_to_merge) > 0:
#     print("Topics_to_merge: ", Topics_to_merge)
#     BERT_model.merge_topics(docs=docs, topics_to_merge=Topics_to_merge)

In [ ]:
# import plotly.graph_objects as go

# # Assuming `BERT_model` is already defined
# topic_info = BERT_model.get_topic_info()
# num_topics = len(topic_info) - 1  # Exclude the outlier topic (-1)

# topn_words = 4  # Number of top words to display for each topic

# labels = []
# parents = []
# values = []

# # Add topics to the sunburst plot
# for i in range(num_topics):
#     topic = BERT_model.get_topic(i)
#     topic_label = f"Topic {i+1}"
#     labels.append(topic_label)
#     parents.append("")
#     values.append(sum([weight for _, weight in topic[:topn_words]]))

#     # Add top words for each topic
#     for word, weight in topic[:topn_words]:
#         labels.append(word)
#         parents.append(topic_label)
#         values.append(weight)

# # Print lists for debugging
# print(labels)
# print(parents)
# print(values)

# # Create the sunburst plot
# fig = go.Figure(go.Sunburst(
#     labels=labels,
#     parents=parents,
#     values=values,
#     branchvalues="total",
#     hovertemplate='<b>%{label}</b><br>Weight: %{value:.2f}<extra></extra>',
# ))

# # Set the layout
# fig.update_layout(
#     title="BERTopic Model Sunburst Plot for Heat Survey",
#     margin=dict(t=50, l=0, r=0, b=0),
# )

# # Show the plot
# fig.show()